In [ ]:
import pandas as pd
import json
import re
import os

from IPython.display import display

pd.set_option("display.max_colwidth", None)

In [ ]:
data_path = os.path.join(os.path.dirname(os.path.abspath(".")), "data/raw/home_depot_data_1_2021_12.csv")
df = pd.read_csv(data_path, index_col=0)

In [ ]:
test_cases_path = os.path.join(os.path.dirname(os.path.abspath(".")), "tests/test_cases_title_description.json")

with open(test_cases_path) as f:
    test_cases = json.load(f)

In [ ]:
select_cols = ['title', 'description', 'brand']

In [ ]:
df[select_cols].head(10)

In [ ]:
df_search = df[select_cols]
df_search['combined_text'] = df_search['title']

In [ ]:
#df_search['combined_text'] = df_search['title'].fillna('') + ' ' + df_search['description'].fillna('')
#df_search['combined_text'] = df_search['title']

## Clean data

The EDA analysis show that we need to perform the following cleaning steps:

- Remove non-ascii characters
- Remove special characters
- Lowercase text

In [ ]:
def remove_non_ascii(text: str) -> str:
    if isinstance(text, str):
        return ''.join(char for char in text if ord(char) < 128)
    return text

In [ ]:
def remove_special_characters(text: str) -> str:
    if isinstance(text, str):
        return ''.join(char for char in text if char.isalnum() or char.isspace())
    return text

In [ ]:
def lowercase_text(text: str) -> str:
    if isinstance(text, str):
        return text.lower()
    return text

In [ ]:
cleaning_steps = [
    remove_non_ascii,
    remove_special_characters,
    lowercase_text
]

result = df_search['combined_text'].copy()
for func in cleaning_steps:
    result = result.apply(func)
df_search['combined_text_clean'] = result

In [ ]:
df_search.head()

## Keyword search engine

### Preprocessing text

Before creating BoW vectors, we need to preprocess the data.

In [ ]:
def remove_stop_words(text: str) -> str:
    from nltk.corpus import stopwords
    stop_words = set(stopwords.words('english'))
    if isinstance(text, str):
        return ' '.join(word for word in text.split() if word not in stop_words)
    return text

In [ ]:
def stemm_words(text: str) -> str:
    from nltk.stem import PorterStemmer
    stemmer = PorterStemmer()
    if isinstance(text, str):
        return ' '.join(stemmer.stem(word) for word in text.split())
    return text

In [ ]:
preprocessing_steps = [
    remove_stop_words,
    stemm_words
]

result = df_search['combined_text_clean'].copy()
for func in preprocessing_steps:
    result = result.apply(func)
    
df_search['combined_text_clean_preprocessed'] = result

In [ ]:
def get_sparse_vectorizer(df: pd.DataFrame, column_name: str):
    from sklearn.feature_extraction.text import TfidfVectorizer
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(df[column_name])
    return tfidf_matrix, vectorizer


In [ ]:
tfidf_matrix, vectorizer = get_sparse_vectorizer(df_search, 'combined_text_clean')


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def keyword_search_engine(query, df, tfidf_matrix, vectorizer, top_n=7):
    # 1. Transform the user's query using the already fitted vectorizer
    query_vec = vectorizer.transform([query])
    
    # 2. Calculate cosine similarity between the query and all documents
    cosine_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # 3. Add the scores to a copy of the dataframe
    results_df = df.copy()
    results_df['similarity_score'] = cosine_scores
    
    # 4. Filter for documents that have a score greater than 0 (relevant matches)
    results_df = results_df[results_df['similarity_score'] > 0]
    
    # 5. Sort by similarity score in descending order
    results_df = results_df.sort_values(by='similarity_score', ascending=False)
    
    # 6. Return the top N results
    return results_df.head(top_n)

In [ ]:
def keyword_preprocess_query(query: str) -> str:
    for func in cleaning_steps + preprocessing_steps:
        query = func(query)
    return query

In [ ]:
def get_keyword_search_results(user_query: str, df: pd.DataFrame, tfidf_matrix, vectorizer, top_n=10):
    preprocessed_query = keyword_preprocess_query(user_query)
    return keyword_search_engine(preprocessed_query, df, tfidf_matrix, vectorizer, top_n)

In [ ]:
def print_search_results(df_search, user_query):
    print(f"Search results for: '{user_query}'\n")
    for _, row in df_search.iterrows():
        print(f"Title: {row['title']}")
        print(f"Description: {row['description']}")
        print(f"Score: {row['similarity_score']:.4f}\n")

In [ ]:
def compute_search_metrics(result_titles, expected_titles):
    found = set(result_titles) & set(expected_titles)
    recall = len(found) / len(expected_titles) if expected_titles else 0
    return recall, found

def display_test_results(search_type, user_query, preprocessed_query, recall, expected_titles, found):
    print(f"{search_type:10} | user_query: {user_query:45} | preprocessed: {preprocessed_query:45} | recall: {recall:.2%}")
    comparison_df = pd.DataFrame({
        'Expected Titles': expected_titles,
        'Found': [title in found for title in expected_titles]
    })
    display(comparison_df.head(len(expected_titles)))

def test_keyword_search_engine(test_cases, search_type, df_search, tfidf_matrix, vectorizer):
    for test in test_cases:
        if test['search_type'] == search_type:
            user_query = test['keyword']
            results_df = get_keyword_search_results(user_query, df_search, tfidf_matrix, vectorizer)
            preprocessed_query = keyword_preprocess_query(user_query)
            result_titles = results_df['title'].tolist()
            expected_titles = [result['title'] for result in test['expected_results']]
            
            recall, found = compute_search_metrics(result_titles, expected_titles)
            display_test_results(search_type, user_query, preprocessed_query, recall, expected_titles, found)

In [ ]:
search_type = "keyword"
test_keyword_search_engine(test_cases, search_type, df_search, tfidf_matrix, vectorizer)

## Semantic search engine

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors

In [ ]:
documents = df_search['title'].tolist()

In [ ]:
print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")
 
print("Encoding documents (this may take a few seconds)...")
document_embeddings = model.encode(documents, show_progress_bar=True)
 
print(f"Created {document_embeddings.shape[0]} embeddings.")
print(f"Embedding dimension: {document_embeddings.shape[1]}")

In [ ]:
print("Creating vector index...")
semantic_search_engine = NearestNeighbors(n_neighbors=5, metric="cosine")
 
semantic_search_engine.fit(document_embeddings)
print("Vector indexis ready!")

In [ ]:
def semantic_search(query, top_k=10):
    # Embed the incoming search query
    query_embedding = model.encode([query])
 
    # Retrieve the closest matches
    distances, indices = semantic_search_engine.kneighbors(query_embedding, n_neighbors=top_k)
 
        
    results_df = [documents[int(indices[0][i])] for i in range(top_k)]
    return results_df

In [ ]:
def test_semantic_search_engine(test_cases):
    for test in test_cases:
        if test['search_type'] == "semantic":
            user_query = test['keyword']
            results_df = semantic_search(user_query)
            expected_titles = [result['title'] for result in test['expected_results']]
            
            recall, found = compute_search_metrics(results_df, expected_titles)
            display_test_results("semantic", user_query, "", recall, expected_titles, found)

In [ ]:
test_semantic_search_engine(test_cases)

## Hybrid search

A semantic search engine combines keyword search (BM25) with semantic search (dense vector embeddings) and merge their scores using Reciprocal Rank Fusion (RRF).

For this implementation, we're using BM25 for keyword search (instead of TF-IDF with cosine similarity).

In [ ]:
keyword_preprocess_query("cordless drill with battery")

In [ ]:
from rank_bm25 import BM25Okapi

corpus = df_search['combined_text_clean_preprocessed'].to_list()
tokenized_corpus = [doc.split(" ") for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
def search_bm25(query, top_k=10):
    preprocessed_query = keyword_preprocess_query(query)
    tokenized_query = preprocessed_query.split()
    
    # Getting scores (lexical relevance to the query) for all documents
    scores = bm25.get_scores(tokenized_query)
    
    # Ranking documents by score
    ranked_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return ranked_indices[:top_k], scores

In [ ]:
ranked_indices, scores = search_bm25("steel toe work boots")
df_search.iloc[ranked_indices]['title'].head()

In [ ]:
from sentence_transformers import util
import torch
doc_embeddings = model.encode(documents, convert_to_tensor=True)
 
def search_semantic(query, top_k=10):
    # Embedding the user's query into a vector
    query_embedding = model.encode(query, convert_to_tensor=True)
    
    # Calculating cosine similarity between the query and all documents
    cosine_scores = util.cos_sim(query_embedding, doc_embeddings)[0]
    
    # Ranking documents by similarity
    ranked_indices = torch.argsort(cosine_scores, descending=True).tolist()
    return ranked_indices[:top_k], cosine_scores.tolist()


In [ ]:
ranked_indices, scores = search_semantic("waterproof jacket for cold weather")
df_search.iloc[ranked_indices]['title'].head()

In [ ]:
def hybrid_search(query, top_k=10):
    # 1. Obtaining the two standalone search rankings
    bm25_ranks, _ = search_bm25(query, top_k=len(documents))
    semantic_ranks, _ = search_semantic(query, top_k=len(documents))
    
    # 2. Applying RRF formula: RRF_score = 1 / (k + rank)
    rrf_scores = {i: 0.0 for i in range(len(documents))}
    k_constant = 60  # The value of 60 is a standard academic convention
    
    # Adding RRF scores from BM25
    for rank, doc_idx in enumerate(bm25_ranks):
        rrf_scores[doc_idx] += 1.0 / (k_constant + rank + 1)
        
    # Adding RRF scores from semantic search
    for rank, doc_idx in enumerate(semantic_ranks):
        rrf_scores[doc_idx] += 1.0 / (k_constant + rank + 1)
    
    # 3. Sorting documents by their final fused RRF score
    final_ranked_indices = sorted(rrf_scores.keys(), key=lambda idx: rrf_scores[idx], reverse=True)
    
    return final_ranked_indices[:top_k], rrf_scores

In [ ]:
ranked_indices, scores = hybrid_search("blue paint for bedroom walls")
df_search.iloc[ranked_indices]['title'].head(n=10)